# Modelos LSTM - Vía de Ingreso Terrestre

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams["figure.figsize"] = (14, 5)

## 1. Carga de Datos

In [ ]:
train = pd.read_csv('../../Datos/train_via_ingreso_terrestre.csv', parse_dates=['Fecha'], index_col='Fecha')
test = pd.read_csv('../../Datos/test_via_ingreso_terrestre.csv', parse_dates=['Fecha'], index_col='Fecha')

print(f"Entrenamiento: {len(train)} meses")
print(f"Prueba: {len(test)} meses")

# Concatenamos para facilitar el procesamiento secuencial
full_data = pd.concat([train, test])
plt.plot(full_data)
plt.axvline(x=train.index[-1], color='r', linestyle='--', label='División Train/Test')
plt.title('Serie Completa')
plt.legend()
plt.show()

## 2. Diferenciación (Estacionariedad) y Normalización

In [ ]:
# Diferenciamos la serie para que sea más estacionaria (las redes LSTM lo prefieren)
serie_diff = full_data.diff().dropna()

# Normalizamos con StandardScaler
train_size = len(train) - 1  # por la diferenciación
scaler = StandardScaler()
scaler.fit(serie_diff.iloc[:train_size])
serie_scaled = scaler.transform(serie_diff)

plt.plot(serie_scaled)
plt.title('Serie Diferenciada y Normalizada')
plt.show()

## 3. Preparación de Secuencias (Supervisada)

In [ ]:
def supervisada(serie, retrasos=1):
    serie_x = []
    serie_y = []
    for i in range(len(serie)-retrasos):
        valor = serie[i:(i+retrasos), 0]
        valor_sig = serie[i+retrasos, 0]
        serie_x.append(valor)
        serie_y.append(valor_sig)
    return np.array(serie_x), np.array(serie_y)

## 4. Entrenamiento de Modelos
### Modelo 1: Configuración Conservadora (Ventana = 12 meses)

In [ ]:
retrasos_m1 = 12
X_m1, y_m1 = supervisada(serie_scaled, retrasos_m1)

# División
# (train_size ya se calculó arriba, al ajustar el scaler)
X_train_m1 = X_m1[:train_size-retrasos_m1]
y_train_m1 = y_m1[:train_size-retrasos_m1]
X_test_m1 = X_m1[train_size-retrasos_m1:]
y_test_m1 = y_m1[train_size-retrasos_m1:]

# Reshape para Keras [muestras, paso, características]
X_train_m1 = np.reshape(X_train_m1, (X_train_m1.shape[0], X_train_m1.shape[1], 1))
X_test_m1 = np.reshape(X_test_m1, (X_test_m1.shape[0], X_test_m1.shape[1], 1))

modelo1 = keras.Sequential([
    layers.Input((retrasos_m1, 1)),
    layers.LSTM(32),
    layers.Dense(1)
])

modelo1.compile(loss='mean_squared_error', optimizer=keras.optimizers.Adam(learning_rate=0.01))
hist1 = modelo1.fit(X_train_m1, y_train_m1, epochs=100, batch_size=8, validation_split=0.2, verbose=0)

plt.plot(hist1.history['loss'], label='Train')
plt.plot(hist1.history['val_loss'], label='Val')
plt.title('Modelo 1 - Pérdida')
plt.legend()
plt.show()

loss_m1 = modelo1.evaluate(X_test_m1, y_test_m1, verbose=0)
print(f"Pérdida en Test (MSE) Modelo 1: {loss_m1:.4f}")

### Modelo 2: Configuración Compleja (Ventana = 24 meses)

In [ ]:
retrasos_m2 = 24
X_m2, y_m2 = supervisada(serie_scaled, retrasos_m2)

X_train_m2 = X_m2[:train_size-retrasos_m2]
y_train_m2 = y_m2[:train_size-retrasos_m2]
X_test_m2 = X_m2[train_size-retrasos_m2:]
y_test_m2 = y_m2[train_size-retrasos_m2:]

X_train_m2 = np.reshape(X_train_m2, (X_train_m2.shape[0], X_train_m2.shape[1], 1))
X_test_m2 = np.reshape(X_test_m2, (X_test_m2.shape[0], X_test_m2.shape[1], 1))

modelo2 = keras.Sequential([
    layers.Input((retrasos_m2, 1)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dense(1)
])

modelo2.compile(loss='mean_squared_error', optimizer=keras.optimizers.Adam(learning_rate=0.005))
hist2 = modelo2.fit(X_train_m2, y_train_m2, epochs=150, batch_size=16, validation_split=0.2, verbose=0)

plt.plot(hist2.history['loss'], label='Train')
plt.plot(hist2.history['val_loss'], label='Val')
plt.title('Modelo 2 - Pérdida')
plt.legend()
plt.show()

loss_m2 = modelo2.evaluate(X_test_m2, y_test_m2, verbose=0)
print(f"Pérdida en Test (MSE) Modelo 2: {loss_m2:.4f}")

## 5. Predicción con el Mejor Modelo

In [ ]:
# Seleccionamos el mejor modelo
if loss_m1 < loss_m2:
    best_model = modelo1
    X_test_best = X_test_m1
    y_test_best = y_test_m1
    retrasos_best = retrasos_m1
    print("Ganador: Modelo 1")
else:
    best_model = modelo2
    X_test_best = X_test_m2
    y_test_best = y_test_m2
    retrasos_best = retrasos_m2
    print("Ganador: Modelo 2")

# Predecimos en test
pred_scaled = best_model.predict(X_test_best)

# Desnormalizamos
pred_diff = scaler.inverse_transform(pred_scaled)


pred_original = []
# El punto de partida es el último valor del set de entrenamiento más el retraso
valores_reales = full_data.values

idx_inicio = train_size + 1

for i in range(len(pred_diff)):
    idx_real = idx_inicio + i
    prev_val = valores_reales[idx_real - 1][0]
    pred_original.append(prev_val + pred_diff[i][0])

pred_original = np.array(pred_original)
real_original = valores_reales[idx_inicio:idx_inicio+len(pred_original)]

# Graficamos Test real vs Predicción
plt.plot(real_original, label='Real (Test)')
plt.plot(pred_original, label='Predicción LSTM')
plt.title('Comparación en Conjunto de Prueba')
plt.legend()
plt.show()

# Métricas finales
rmse_final = np.sqrt(mean_squared_error(real_original, pred_original))
mae_final = mean_absolute_error(real_original, pred_original)
print(f"RMSE Final en Test: {rmse_final:.2f}")
print(f"MAE Final en Test: {mae_final:.2f}")